In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
import random
from PIL import Image
import timm
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import random

In [2]:
def set_random_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = "42"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_random_seed()

In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, transform=None, img_dir='/kaggle/input/datasets/yuulind/pa-100k/PA-100K/data/'):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.img_labels)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = torch.from_numpy(self.img_labels.iloc[idx, 1:].values.astype(np.float32))
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((384, 192)),  # maintain pedestrian shape, taller than wide
    transforms.RandomHorizontalFlip(),  # common for pedestrian symmetry
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((384, 192), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor()
])

train_dataset = CustomImageDataset('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/train.csv', train_transform)
val_dataset = CustomImageDataset('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/val.csv', val_transform)
test_dataset = CustomImageDataset('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/test.csv', val_transform)

In [5]:
r = pd.read_csv('/kaggle/input/datasets/yuulind/pa-100k/PA-100K/train.csv').iloc[:,1:].values.mean(0)
r = torch.from_numpy(r).type(torch.float32)

In [6]:
# Common DataLoader kwargs
loader_kwargs = {
    'batch_size': 16,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
}

train_loader = DataLoader(
    train_dataset, 
    shuffle=True,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset, 
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset, 
    shuffle=False,
    **loader_kwargs
)

In [7]:
def compute_mFive(y_true, y_pred):
    y_true = np.array(y_true, dtype='float32')
    y_pred = np.array(y_pred, dtype='float32')
    
    # ---- mA (mean per-label accuracy) ----
    L = y_true.shape[1]
    TP = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    TN = np.sum((y_true == 0) & (y_pred == 0), axis=0)
    P = np.sum(y_true == 1, axis=0)
    N = np.sum(y_true == 0, axis=0)

    tp_over_p = np.where(P == 0, 1.0, TP / P)
    tn_over_n = np.where(N == 0, 1.0, TN / N)

    mA = (1 / (2 * L)) * np.sum(tp_over_p + tn_over_n)

    # ---- Instance-level metrics ----
    intersection = np.logical_and(y_true, y_pred).sum(axis=1)
    union = np.logical_or(y_true, y_pred).sum(axis=1)
    pred_sum = y_pred.sum(axis=1)
    true_sum = y_true.sum(axis=1)

    Accuracy = np.mean(intersection / union)
    Precision = np.mean(intersection / pred_sum)
    Recall = np.mean(intersection / true_sum)
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # ---- mFive ----
    mFive = np.mean([mA, Accuracy, Precision, Recall, F1])

    return {
        'mA': mA,
        'Accuracy': Accuracy,
        'Precision': Precision,
        'Recall': Recall,
        'F1': F1,
        'mFive': mFive
    }

In [8]:
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=False)
    for i, (images, labels) in enumerate(pbar, 1):
        images = images.to(device)
        labels = torch.abs(labels-torch.rand(labels.shape) * 0.2)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        labels = (labels >= 0.5).float()
        correct += (preds == labels).float().sum().item()
        total += labels.numel()

        pbar.set_postfix({
            'Avg Loss': f'{running_loss / i:.4f}',
            'Avg Acc': f'{correct / total:.4f}'
        })
        
    scheduler.step()
    return running_loss / len(dataloader), correct / total


def validate_one_epoch(model, dataloader, criterion, device, epoch, num_epochs):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'Avg Loss': f'{running_loss / i:.4f}',
                'Avg Acc': f'{correct / total:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)
    return running_loss / len(dataloader), correct / total, metrics


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20):
    model.to(device)
    best_val_mfive = 0
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, epoch, num_epochs)
        val_loss, val_acc, metrics = validate_one_epoch(model, val_loader, criterion, device, epoch, num_epochs)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Train Loss:     {train_loss:.4f} - Acc: {train_acc:.4f}")
        print(f"  Val Loss:{val_loss:.4f} - Val Acc: {val_acc:.4f}")
        print(f"  Val mFive:      {metrics['mFive']:.4f}")
        print(f"  Val Acc:         {metrics['Accuracy']:.4f}")
        print(f"  Val mA:         {metrics['mA']:.4f}")
        print(f"  Val F1:         {metrics['F1']:.4f}")
        print(f"  Val Precision:  {metrics['Precision']:.4f}")
        print(f"  Val Recall:     {metrics['Recall']:.4f}")
        print("-" * 60)

        if metrics["mFive"] > best_val_mfive:
            best_val_mfive = metrics["mFive"]
            torch.save(model.state_dict(), 'best_weight.pth')
            print(f"✅ Best model saved (mFive: {best_val_mfive:.4f})")

    return model, train_losses, val_losses

def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(test_loader, desc="Testing", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            avg_loss = running_loss / i
            avg_acc = correct / total

            pbar.set_postfix({
                'Avg Loss': f'{avg_loss:.4f}',
                'Avg Acc': f'{avg_acc:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)

    print(f"\nTest Summary:")
    print(f"  Test Loss:     {running_loss / len(test_loader):.4f}")
    print(f"  Test mFive:    {metrics['mFive']:.4f}")
    print(f"  Test Accuracy: {metrics['Accuracy']:.4f}")
    print(f"  Test mA:       {metrics['mA']:.4f}")
    print(f"  Test F1:       {metrics['F1']:.4f}")
    print(f"  Test Precision:{metrics['Precision']:.4f}")
    print(f"  Test Recall:   {metrics['Recall']:.4f}")
    print("-" * 60)

    return metrics

In [9]:
class Eff_Baseline(nn.Module):
    def __init__(self, num_classes):
        super(Eff_Baseline, self).__init__()
        backbone = models.efficientnet_v2_m(weights="DEFAULT")
        self.backbone = backbone.features  
        self.pool = nn.AdaptiveAvgPool2d(1)  
        self.classifier = nn.Linear(1280, num_classes)  

    def forward(self, x):
        x = self.backbone(x)          
        x = self.pool(x)              
        x = torch.flatten(x, 1)       
        x = self.classifier(x)        
        return x

In [10]:
class WeightedBCELoss(nn.Module):
    def __init__(self, r):
        super(WeightedBCELoss, self).__init__()
        self.r = r 

    def forward(self, logits, targets): 
        probs = torch.sigmoid(logits) 
        pos_weights = torch.exp(1 - self.r).expand_as(targets)
        neg_weights = torch.exp(self.r).expand_as(targets)
        weighted_bce = pos_weights * targets * torch.log(probs + 1e-7) + neg_weights * (1 - targets) * torch.log(1 - probs + 1e-7)
        return (-weighted_bce).mean()


In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model=Eff_Baseline(num_classes=26)

criterion = WeightedBCELoss(r.to(device)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
    
# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 211MB/s]


In [12]:
model, train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20)


Epoch 1/20 Summary:
  Train Loss:     0.6910 - Acc: 0.9487
  Val Loss:0.4018 - Val Acc: 0.9410
  Val mFive:      0.8468
  Val Acc:         0.7917
  Val mA:         0.8205
  Val F1:         0.8740
  Val Precision:  0.8717
  Val Recall:     0.8763
------------------------------------------------------------
✅ Best model saved (mFive: 0.8468)



Epoch 2/20 Summary:
  Train Loss:     0.6535 - Acc: 0.9688
  Val Loss:0.3891 - Val Acc: 0.9431
  Val mFive:      0.8527
  Val Acc:         0.7994
  Val mA:         0.8276
  Val F1:         0.8788
  Val Precision:  0.8708
  Val Recall:     0.8869
------------------------------------------------------------
✅ Best model saved (mFive: 0.8527)



Epoch 3/20 Summary:
  Train Loss:     0.6314 - Acc: 0.9797
  Val Loss:0.3750 - Val Acc: 0.9466
  Val mFive:      0.8622
  Val Acc:         0.8118
  Val mA:         0.8400
  Val F1:         0.8864
  Val Precision:  0.8785
  Val Recall:     0.8943
------------------------------------------------------------
✅ Best model saved (mFive: 0.8622)



Epoch 4/20 Summary:
  Train Loss:     0.6251 - Acc: 0.9831
  Val Loss:0.3781 - Val Acc: 0.9466
  Val mFive:      0.8613
  Val Acc:         0.8111
  Val mA:         0.8378
  Val F1:         0.8858
  Val Precision:  0.8796
  Val Recall:     0.8921
------------------------------------------------------------



Epoch 5/20 Summary:
  Train Loss:     0.6211 - Acc: 0.9851
  Val Loss:0.3733 - Val Acc: 0.9471
  Val mFive:      0.8624
  Val Acc:         0.8129
  Val mA:         0.8384
  Val F1:         0.8869
  Val Precision:  0.8802
  Val Recall:     0.8936
------------------------------------------------------------
✅ Best model saved (mFive: 0.8624)


...


Epoch 16/20 Summary:
  Train Loss:     0.6195 - Acc: 0.9859
  Val Loss:0.3727 - Val Acc: 0.9473
  Val mFive:      0.8632
  Val Acc:         0.8137
  Val mA:         0.8396
  Val F1:         0.8875
  Val Precision:  0.8813
  Val Recall:     0.8939
------------------------------------------------------------



Epoch 17/20 Summary:
  Train Loss:     0.6199 - Acc: 0.9858
  Val Loss:0.3739 - Val Acc: 0.9471
  Val mFive:      0.8629
  Val Acc:         0.8132
  Val mA:         0.8396
  Val F1:         0.8871
  Val Precision:  0.8807
  Val Recall:     0.8937
------------------------------------------------------------



Epoch 18/20 Summary:
  Train Loss:     0.6198 - Acc: 0.9858
  Val Loss:0.3725 - Val Acc: 0.9473
  Val mFive:      0.8633
  Val Acc:         0.8139
  Val mA:         0.8396
  Val F1:         0.8876
  Val Precision:  0.8814
  Val Recall:     0.8938
------------------------------------------------------------



Epoch 19/20 Summary:
  Train Loss:     0.6199 - Acc: 0.9858
  Val Loss:0.3724 - Val Acc: 0.9475
  Val mFive:      0.8632
  Val Acc:         0.8140
  Val mA:         0.8391
  Val F1:         0.8876
  Val Precision:  0.8821
  Val Recall:     0.8932
------------------------------------------------------------



Epoch 20/20 Summary:
  Train Loss:     0.6199 - Acc: 0.9859
  Val Loss:0.3734 - Val Acc: 0.9472
  Val mFive:      0.8627
  Val Acc:         0.8134
  Val mA:         0.8381
  Val F1:         0.8872
  Val Precision:  0.8811
  Val Recall:     0.8935
------------------------------------------------------------


In [13]:
model.load_state_dict(torch.load('best_weight.pth'))
metrics = test_model(model, test_loader, criterion, device)


Test Summary:
  Test Loss:     0.3676
  Test mFive:    0.8631
  Test Accuracy: 0.8122
  Test mA:       0.8413
  Test F1:       0.8872
  Test Precision:0.8794
  Test Recall:   0.8952
------------------------------------------------------------
